# 04 — Modelling Experiments

Phase 4, notebook 1 of 3. Baselines, a time-based train/validation/test
split, an empirical comparison of imbalance-handling techniques (per
[DECISIONS.md#imbalance-is-designed-in](../DECISIONS.md#imbalance-is-designed-in) —
class weighting vs. SMOTE vs. threshold tuning, "none of the three is
picked in advance"), Optuna hyperparameter tuning nested inside
`GroupKFold(groups=customer_id)` (per
[DECISIONS.md#leakage-mitigations](../DECISIONS.md#leakage-mitigations)),
and MLflow experiment logging.

In [1]:
import os
import sys
sys.path.insert(0, "../src")

import decimal
import warnings

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import text

from churn.config import PROJECT_ROOT, get_app_config

os.chdir(PROJECT_ROOT)

from churn.db.engine import get_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore", category=FutureWarning)


def coerce_decimal_columns(frame: pd.DataFrame) -> pd.DataFrame:
    for c in frame.columns:
        if frame[c].dtype == object and frame[c].map(lambda v: isinstance(v, decimal.Decimal)).any():
            frame[c] = pd.to_numeric(frame[c], errors="coerce")
    return frame


cfg = get_app_config()
RANDOM_SEED = cfg.synthetic.random_seed
engine = get_engine()

panel = pd.read_sql(
    text("SELECT * FROM feature_snapshots WHERE feature_set_version = :v"),
    engine,
    params={"v": cfg.feature_set.version},
)
panel["reference_date"] = pd.to_datetime(panel["reference_date"])
panel = coerce_decimal_columns(panel)

snapshot_ts = pd.Timestamp(cfg.data.snapshot_date)
modeling_panel = panel[panel["reference_date"] < snapshot_ts].copy()
print(f"Modeling panel (last cohort dropped): {modeling_panel.shape}")

with open(PROJECT_ROOT / "config" / "selected_features_v1.yaml", encoding="utf-8") as fh:
    feature_artifact = yaml.safe_load(fh)
selected_features = feature_artifact["selected_features"]
print(f"Selected features ({len(selected_features)}):", selected_features)

Modeling panel (last cohort dropped): (82343, 39)
Selected features (12): ['contract_type', 'online_security', 'tech_support', 'tenure_months', 'tenure_bucket', 'frequency_charges_90d', 'frequency_tickets_30d', 'frequency_tickets_90d', 'internet_service', 'online_backup', 'payment_method', 'monetary_charges_90d']


## 1. Time-based train / validation / test split

Per `config/config.yaml`'s `data:` block (`train_months=12,
validation_months=3, test_months=3`), applied as calendar-month cutoffs
from `panel_start_date`. Note the resulting test window has only 2
cohorts, not 3: the third nominal test month (`2024-06-01`) is exactly
the structurally-censored cohort already dropped above — see
[DECISIONS.md#last-cohort-dropped-from-modeling-panel](../DECISIONS.md#last-cohort-dropped-from-modeling-panel).
Flagged here explicitly rather than silently absorbed.

In [2]:
from dateutil.relativedelta import relativedelta

panel_start = pd.Timestamp(cfg.data.panel_start_date)
train_end = panel_start + relativedelta(months=cfg.data.train_months)          # exclusive
val_end = train_end + relativedelta(months=cfg.data.validation_months)         # exclusive
test_end = val_end + relativedelta(months=cfg.data.test_months)                # exclusive (nominal)

train_df = modeling_panel[modeling_panel["reference_date"] < train_end]
val_df = modeling_panel[(modeling_panel["reference_date"] >= train_end) & (modeling_panel["reference_date"] < val_end)]
test_df = modeling_panel[(modeling_panel["reference_date"] >= val_end) & (modeling_panel["reference_date"] < test_end)]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cohorts = sorted(d["reference_date"].dt.date.unique())
    print(f"{name}: {len(d):>6} rows | {d['customer_id'].nunique():>5} customers | "
          f"cohorts {cohorts[0]}..{cohorts[-1]} ({len(cohorts)} months) | "
          f"positive rate {100*d['label_churn_next_30d'].mean():.2f}%")

assert test_df["reference_date"].dt.date.max() < snapshot_ts.date(), "Test window reaches the censored cohort."
overlap = set(train_df.customer_id) & set(test_df.customer_id)
print(f"\nCustomers appearing in both train and test (expected -- same customer, different months; "
      f"GroupKFold only matters WITHIN a split, for the tuning CV below): {len(overlap)}")

train:  57021 rows |  5812 customers | cohorts 2023-01-01..2023-12-01 (12 months) | positive rate 1.67%
val:  14994 rows |  5201 customers | cohorts 2024-01-01..2024-03-01 (3 months) | positive rate 1.57%
test:  10328 rows |  5327 customers | cohorts 2024-04-01..2024-05-01 (2 months) | positive rate 1.58%



Customers appearing in both train and test (expected -- same customer, different months; GroupKFold only matters WITHIN a split, for the tuning CV below): 4657


## 2. Preprocessing

One shared `ColumnTransformer` (one-hot categoricals, scale numerics) for
every model below, including the linear baseline -- standard practice and
harmless for the tree-based models.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_features = [
    c for c in selected_features
    if not (pd.api.types.is_numeric_dtype(modeling_panel[c]) and not pd.api.types.is_bool_dtype(modeling_panel[c]))
]
num_features = [c for c in selected_features if c not in cat_features]
print("categorical:", cat_features)
print("numeric:", num_features)

def make_preprocessor() -> ColumnTransformer:
    # A fresh instance per pipeline, always -- sklearn Pipeline does not
    # clone its steps, so two pipelines sharing one ColumnTransformer
    # silently corrupt each other's fitted state the moment the second
    # one is fit (e.g. gbm_smote's fit-on-resampled-data would leave
    # gbm_plain's *already-fitted* preprocessor rescaled out from under
    # it, corrupting any later reuse of gbm_plain -- a real bug caught
    # by review after this notebook first shipped).
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", StandardScaler(), num_features),
    ])

X_train, y_train, groups_train = train_df[selected_features], train_df["label_churn_next_30d"].astype(int), train_df["customer_id"]
X_val, y_val = val_df[selected_features], val_df["label_churn_next_30d"].astype(int)
X_test, y_test = test_df[selected_features], test_df["label_churn_next_30d"].astype(int)

categorical: ['contract_type', 'online_security', 'tech_support', 'tenure_bucket', 'internet_service', 'online_backup', 'payment_method']
numeric: ['tenure_months', 'frequency_charges_90d', 'frequency_tickets_30d', 'frequency_tickets_90d', 'monetary_charges_90d']


## 3. MLflow setup

In [4]:
import mlflow

mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)
mlflow.set_experiment(cfg.mlflow.experiment_name)
print("tracking:", mlflow.get_tracking_uri(), "| experiment:", cfg.mlflow.experiment_name)

2026/09/22 11:49:10 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\11624\OneDrive - Indian School of Business\Documents\personal\ml_proj\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


tracking: http://localhost:5000 | experiment: churn-prediction


## 4. Baselines

`DummyClassifier(strategy="prior")` gives the formal no-skill PR-AUC
(exactly the validation positive rate) as a reference point; logistic
regression is the first model that actually uses the features.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

results = []


def fit_eval_log(name, pipeline, X_tr, y_tr, X_ev, y_ev, extra_params=None, fit_kwargs=None):
    pipeline.fit(X_tr, y_tr, **(fit_kwargs or {}))
    proba = pipeline.predict_proba(X_ev)[:, 1]
    pr_auc = average_precision_score(y_ev, proba)
    with mlflow.start_run(run_name=name):
        mlflow.set_tag("phase", "4-modelling-experiments")
        mlflow.log_params(extra_params or {})
        mlflow.log_metric("val_pr_auc", pr_auc)
        mlflow.log_metric("val_positive_rate", float(y_ev.mean()))
    results.append({"model": name, "val_pr_auc": pr_auc})
    print(f"{name:35s} val PR-AUC = {pr_auc:.4f}")
    return pipeline, pr_auc


dummy = SkPipeline([("prep", make_preprocessor()), ("clf", DummyClassifier(strategy="prior"))])
fit_eval_log("baseline_dummy_prior", dummy, X_train, y_train, X_val, y_val)

logreg = SkPipeline([("prep", make_preprocessor()), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED))])
fit_eval_log("baseline_logreg_balanced", logreg, X_train, y_train, X_val, y_val)

🏃 View run baseline_dummy_prior at: http://localhost:5000/#/experiments/1/runs/7dfd8d51ea0f4edfbe9010d8d1ff1317
🧪 View experiment at: http://localhost:5000/#/experiments/1


baseline_dummy_prior                val PR-AUC = 0.0157


🏃 View run baseline_logreg_balanced at: http://localhost:5000/#/experiments/1/runs/a44da5addcd04d2eb7e422bd2518d80b
🧪 View experiment at: http://localhost:5000/#/experiments/1
baseline_logreg_balanced            val PR-AUC = 0.1281


(Pipeline(steps=[('prep',
                  ColumnTransformer(transformers=[('cat',
                                                   OneHotEncoder(handle_unknown='ignore'),
                                                   ['contract_type',
                                                    'online_security',
                                                    'tech_support',
                                                    'tenure_bucket',
                                                    'internet_service',
                                                    'online_backup',
                                                    'payment_method']),
                                                  ('num', StandardScaler(),
                                                   ['tenure_months',
                                                    'frequency_charges_90d',
                                                    'frequency_tickets_30d',
                                    

## 5. Imbalance-handling comparison (HistGradientBoostingClassifier)

`HistGradientBoostingClassifier` (scikit-learn, already a dependency)
rather than adding `xgboost`/`lightgbm` -- a native, histogram-based
boosting model (LightGBM-style binned splits) with no new dependency,
consistent with this project's preference for hand-rolling over pulling
in another package when the built-in tool is adequate (same reasoning as
the Boruta shadow-feature implementation in Phase 3). Chosen over the
classic `GradientBoostingClassifier` specifically for speed: exact-split
boosting on a ~66k-row training fold, repeated across dozens of Optuna
trials, would take far longer than the histogram-based version.

Three variants, per
[DECISIONS.md#imbalance-is-designed-in](../DECISIONS.md#imbalance-is-designed-in):
**class weighting** (`sample_weight` -- `HistGradientBoostingClassifier`
has no `class_weight` param, so this is the equivalent lever), **SMOTE**
(`SMOTENC`, since several selected features are categorical -- plain
`SMOTE` would linearly interpolate one-hot columns into meaningless
fractional values; `SMOTENC` is applied to the *raw* frame, before
one-hot encoding, and only ever touches the training fold), and **plain**
(no rebalancing, the base case **threshold tuning** is applied to below).

In [6]:
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

GBM_DEFAULTS = dict(max_iter=150, max_depth=3, learning_rate=0.1, random_state=RANDOM_SEED)
cat_mask = [c in cat_features for c in selected_features]

In [7]:
# -- plain (no rebalancing) -- its own cell so a slow/hung fit here (this
# machine's timing has been erratic -- OneDrive sync and/or endpoint
# security scanning joblib's temp files, not the code) is visible on its
# own rather than swallowed inside one big multi-fit cell.
gbm_plain = SkPipeline([("prep", make_preprocessor()), ("clf", HistGradientBoostingClassifier(**GBM_DEFAULTS))])
_, pr_auc_plain = fit_eval_log("gbm_plain", gbm_plain, X_train, y_train, X_val, y_val, GBM_DEFAULTS)

🏃 View run gbm_plain at: http://localhost:5000/#/experiments/1/runs/51c220994d65423d971eeb7dff9f7bbe
🧪 View experiment at: http://localhost:5000/#/experiments/1


gbm_plain                           val PR-AUC = 0.0529


In [8]:
# -- class weighting (sample_weight) --
gbm_classweight = SkPipeline([("prep", make_preprocessor()), ("clf", HistGradientBoostingClassifier(**GBM_DEFAULTS))])
sw = compute_sample_weight("balanced", y_train)
_, pr_auc_classweight = fit_eval_log(
    "gbm_class_weighting", gbm_classweight, X_train, y_train, X_val, y_val,
    GBM_DEFAULTS, fit_kwargs={"clf__sample_weight": sw},
)

🏃 View run gbm_class_weighting at: http://localhost:5000/#/experiments/1/runs/c8c6ff32f1954ff99e3f18e9b59699e6
🧪 View experiment at: http://localhost:5000/#/experiments/1
gbm_class_weighting                 val PR-AUC = 0.0483


In [9]:
# -- SMOTE (SMOTENC, train fold only) --
gbm_smote = ImbPipeline([
    ("smote", SMOTENC(categorical_features=cat_mask, random_state=RANDOM_SEED)),
    ("prep", make_preprocessor()),
    ("clf", HistGradientBoostingClassifier(**GBM_DEFAULTS)),
])
_, pr_auc_smote = fit_eval_log("gbm_smote", gbm_smote, X_train, y_train, X_val, y_val, GBM_DEFAULTS)

🏃 View run gbm_smote at: http://localhost:5000/#/experiments/1/runs/a4529a861128427b81504ac72b1e6f0b
🧪 View experiment at: http://localhost:5000/#/experiments/1
gbm_smote                           val PR-AUC = 0.0606


In [10]:
technique_scores = {"plain": pr_auc_plain, "class_weighting": pr_auc_classweight, "smote": pr_auc_smote}
winning_technique = max(technique_scores, key=technique_scores.get)
print()
print("Technique comparison (val PR-AUC):", technique_scores)
print("Winner:", winning_technique)


Technique comparison (val PR-AUC): {'plain': 0.05292596235148419, 'class_weighting': 0.04825893845461123, 'smote': 0.060573126562861394}
Winner: smote


## 6. Threshold tuning (the third technique)

PR-AUC is threshold-independent by construction -- it cannot be "won" or
"lost" by moving a decision threshold. Threshold tuning is evaluated the
way it actually operates: applied to the plain (no-rebalancing) model,
comparing F1 at the default 0.5 cutoff against F1 at the threshold that
maximizes F1 on the validation set. Reported alongside the PR-AUC
comparison above, not folded into it, because they answer different
questions -- ranking quality vs. operating-point quality. Cost-based
threshold selection (using the actual business economics) is Notebook
05's job; this is the plain statistical version.

In [11]:
from sklearn.metrics import f1_score, precision_recall_curve

val_proba_plain = gbm_plain.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, val_proba_plain)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = int(np.nanargmax(f1_scores[:-1]))  # last point has no corresponding threshold
best_threshold = thresholds[best_idx]

f1_default = f1_score(y_val, (val_proba_plain >= 0.5).astype(int))
f1_tuned = f1_score(y_val, (val_proba_plain >= best_threshold).astype(int))

with mlflow.start_run(run_name="gbm_plain_threshold_tuning"):
    mlflow.set_tag("phase", "4-modelling-experiments")
    mlflow.log_metric("f1_at_0.5", f1_default)
    mlflow.log_metric("f1_at_tuned_threshold", f1_tuned)
    mlflow.log_metric("tuned_threshold", float(best_threshold))

print(f"F1 @ threshold=0.5:            {f1_default:.4f}")
print(f"F1 @ tuned threshold={best_threshold:.3f}:  {f1_tuned:.4f}")
print(f"Improvement: {f1_tuned - f1_default:+.4f}")

🏃 View run gbm_plain_threshold_tuning at: http://localhost:5000/#/experiments/1/runs/0c35e7cc86084898996b67a3853fbdff
🧪 View experiment at: http://localhost:5000/#/experiments/1
F1 @ threshold=0.5:            0.0229
F1 @ tuned threshold=0.096:  0.1183
Improvement: +0.0953


## 7. Optuna hyperparameter tuning

Tunes the winning technique's `HistGradientBoostingClassifier`, using
`GroupKFold(groups=customer_id)` **nested inside the training window
only** -- the validation and test date-ranges are never touched during
tuning, per
[DECISIONS.md#leakage-mitigations](../DECISIONS.md#leakage-mitigations)
point 4.

In [12]:
import optuna
from sklearn.model_selection import GroupKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 12
N_SPLITS = 2
gkf = GroupKFold(n_splits=N_SPLITS)


def make_pipeline(params):
    clf = HistGradientBoostingClassifier(random_state=RANDOM_SEED, **params)
    if winning_technique == "smote":
        return ImbPipeline([
            ("smote", SMOTENC(categorical_features=cat_mask, random_state=RANDOM_SEED)),
            ("prep", make_preprocessor()), ("clf", clf),
        ]), {}
    if winning_technique == "class_weighting":
        return SkPipeline([("prep", make_preprocessor()), ("clf", clf)]), {"clf__sample_weight": "AUTO"}
    return SkPipeline([("prep", make_preprocessor()), ("clf", clf)]), {}


def objective(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 50, 150),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 1.0),
    }
    scores = []
    for fold_train_idx, fold_val_idx in gkf.split(X_train, y_train, groups=groups_train):
        Xtr, Xva = X_train.iloc[fold_train_idx], X_train.iloc[fold_val_idx]
        ytr, yva = y_train.iloc[fold_train_idx], y_train.iloc[fold_val_idx]
        pipe, fit_kwargs = make_pipeline(params)
        if fit_kwargs.get("clf__sample_weight") == "AUTO":
            fit_kwargs = {"clf__sample_weight": compute_sample_weight("balanced", ytr)}
        pipe.fit(Xtr, ytr, **fit_kwargs)
        proba = pipe.predict_proba(Xva)[:, 1]
        scores.append(average_precision_score(yva, proba))
    return float(np.mean(scores))


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print(f"Best CV PR-AUC (mean over {N_SPLITS} GroupKFold folds): {study.best_value:.4f}")
print("Best params:", study.best_params)

Best CV PR-AUC (mean over 2 GroupKFold folds): 0.1101
Best params: {'max_iter': 101, 'max_depth': 2, 'learning_rate': 0.15173165843487205, 'min_samples_leaf': 33, 'l2_regularization': 0.5779324729105735}


## 8. Final tuned model: refit on the full training set, evaluate on validation

In [13]:
best_pipe, fit_kwargs = make_pipeline(study.best_params)
if fit_kwargs.get("clf__sample_weight") == "AUTO":
    fit_kwargs = {"clf__sample_weight": compute_sample_weight("balanced", y_train)}
best_pipe.fit(X_train, y_train, **fit_kwargs)
val_proba_tuned = best_pipe.predict_proba(X_val)[:, 1]
val_pr_auc_tuned = average_precision_score(y_val, val_proba_tuned)

with mlflow.start_run(run_name=f"gbm_{winning_technique}_optuna_tuned"):
    mlflow.set_tag("phase", "4-modelling-experiments")
    mlflow.set_tag("imbalance_technique", winning_technique)
    mlflow.log_params(study.best_params)
    mlflow.log_metric("cv_pr_auc_mean", study.best_value)
    mlflow.log_metric("val_pr_auc", val_pr_auc_tuned)
    trials_path = PROJECT_ROOT / "artifacts" / "optuna_trials_phase4.csv"
    trials_path.parent.mkdir(exist_ok=True)
    study.trials_dataframe().to_csv(trials_path, index=False)
    mlflow.log_artifact(str(trials_path))

results.append({"model": f"gbm_{winning_technique}_optuna_tuned", "val_pr_auc": val_pr_auc_tuned})
print(f"Tuned {winning_technique} GBM val PR-AUC: {val_pr_auc_tuned:.4f} "
      f"(untuned {winning_technique} was {technique_scores[winning_technique]:.4f})")

🏃 View run gbm_smote_optuna_tuned at: http://localhost:5000/#/experiments/1/runs/351ffee50bdb4dfcb20da743d287f864
🧪 View experiment at: http://localhost:5000/#/experiments/1
Tuned smote GBM val PR-AUC: 0.0609 (untuned smote was 0.0606)


## 9. Summary

Honest result, not the expected one: the plain regularized logistic
regression baseline beats every `HistGradientBoostingClassifier` variant
tried here, tuned or not. Reported as found, not smoothed over -- same
principle as Notebook 03's engineered features not making the final
selection. Plausible reason: only 12 features (7 of them low-cardinality
categoricals contributing little nonlinear structure for a tree ensemble
to exploit), a training set with ~1.7% positives, and no interaction
features (Notebook 03's engineered ratio/interaction features weren't
selected) -- exactly the regime where a well-regularized linear model's
lower variance tends to win over a boosted ensemble's extra flexibility.

In [14]:
results_df = pd.DataFrame(results).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
display(results_df)

overall_best = results_df.iloc[0]
print(f"Overall best model (val PR-AUC): {overall_best['model']} = {overall_best['val_pr_auc']:.4f}")
print(f"Best GBM variant (tuned {winning_technique}): {val_pr_auc_tuned:.4f}")
if overall_best["model"] != f"gbm_{winning_technique}_optuna_tuned":
    print(f"\n*** The GBM line of experiments (imbalance-technique comparison + Optuna tuning) never beat "
          f"the logistic regression baseline. This is the real result, carried into Notebook 05 as-is. ***")

print("\nWinning imbalance-handling technique among GBM variants specifically (by val PR-AUC): ", winning_technique)
print("Optuna best hyperparameters (for that GBM variant):", study.best_params)
print(f"\nThreshold tuning (applied to the plain GBM, separately from the technique comparison above):")
print(f"  F1 @ 0.5 = {f1_default:.4f}  ->  F1 @ tuned threshold {best_threshold:.3f} = {f1_tuned:.4f}")
print("\nCarried into Notebook 05: refit the overall best model "
      f"({overall_best['model']}) on train+val, evaluate on the held-out test set -- "
      "the tuned GBM is reported alongside it for completeness, not as the primary model.")

,model,val_pr_auc
0,baseline_logreg_balanced,0.128089
1,gbm_smote_optuna_tuned,0.060896
2,gbm_smote,0.060573
3,gbm_plain,0.052926
4,gbm_class_weighting,0.048259
5,baseline_dummy_prior,0.015740


Overall best model (val PR-AUC): baseline_logreg_balanced = 0.1281
Best GBM variant (tuned smote): 0.0609

*** The GBM line of experiments (imbalance-technique comparison + Optuna tuning) never beat the logistic regression baseline. This is the real result, carried into Notebook 05 as-is. ***

Winning imbalance-handling technique among GBM variants specifically (by val PR-AUC):  smote
Optuna best hyperparameters (for that GBM variant): {'max_iter': 101, 'max_depth': 2, 'learning_rate': 0.15173165843487205, 'min_samples_leaf': 33, 'l2_regularization': 0.5779324729105735}

Threshold tuning (applied to the plain GBM, separately from the technique comparison above):
  F1 @ 0.5 = 0.0229  ->  F1 @ tuned threshold 0.096 = 0.1183

Carried into Notebook 05: refit the overall best model (baseline_logreg_balanced) on train+val, evaluate on the held-out test set -- the tuned GBM is reported alongside it for completeness, not as the primary model.
